In [1]:
!nvidia-smi
!apt-get update -y
!apt-get install -y curl zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Ollama'nın uyanması için bekliyoruz
time.sleep(10)
print("Ollama başarıyla başlatıldı ✅")

Sun Apr 19 04:34:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!ollama pull gemma3:4b
!ollama pull embeddinggemma
!curl http://127.0.0.1:11434/api/tags
!python --version
!which ollama



{"models":[{"name":"embeddinggemma:latest","model":"embeddinggemma:latest","modified_at":"2026-04-19T04:36:05.916892342Z","size":621875917,"digest":"85462619ee721b466c5927d109d4cb765861907d5417b9109caebc4e614679f1","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"307.58M","quantization_level":"BF16"}},{"name":"gemma3:4b","model":"gemma3:4b","modified_at":"2026-04-19T04:35:46.957399518Z","size":3338801804,"digest":"a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"4.3B","quantization_level":"Q4_K_M"}}]}Python 3.12.13
/usr/local/bin/ollama


In [3]:
import json
import math
import os
import pickle
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
CHAT_MODEL = "gemma3:4b"
EMBED_MODEL = "embeddinggemma"

JSONL_PATH = "/content/all_datasets_combined.jsonl"
INDEX_PATH = "/content/rag_index.pkl"

TOP_K_CANDIDATES = 80
TOP_K_CONTEXT = 8
EMBED_BATCH_SIZE = 64

@dataclass
class SearchHit:
    score: float
    record: Dict[str, Any]

class OllamaClient:
    def __init__(self, base_url: str = OLLAMA_URL):
        self.base_url = base_url.rstrip("/")

    def embed_one(self, text: str) -> List[float]:
        resp = requests.post(
            f"{self.base_url}/api/embed",
            json={
                "model": EMBED_MODEL,
                "input": text,
                "truncate": True,
                "keep_alive": "15m",
            },
            timeout=180,
        )
        resp.raise_for_status()
        data = resp.json()
        return data["embeddings"][0]

    def embed_many(self, texts: List[str]) -> List[List[float]]:
        resp = requests.post(
            f"{self.base_url}/api/embed",
            json={
                "model": EMBED_MODEL,
                "input": texts,
                "truncate": True,
                "keep_alive": "15m",
            },
            timeout=600,
        )
        resp.raise_for_status()
        data = resp.json()
        return data["embeddings"]

    def generate(self, prompt: str) -> str:
        resp = requests.post(
            f"{self.base_url}/api/generate",
            json={
                "model": CHAT_MODEL,
                "prompt": prompt,
                "stream": False,
                "keep_alive": "15m",
                "options": {
                    "temperature": 0.1,
                    "num_predict": 400,
                }
            },
            timeout=300,
        )
        resp.raise_for_status()
        return resp.json()["response"]

def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text: str) -> List[str]:
    text = normalize_text(text).lower()
    text = (
        text.replace("ı", "i")
        .replace("İ", "i")
        .replace("ş", "s")
        .replace("ğ", "g")
        .replace("ü", "u")
        .replace("ö", "o")
        .replace("ç", "c")
    )
    return re.findall(r"[a-zA-Z0-9_]+", text)

def cosine_similarity(a: List[float], b: List[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)

def keyword_overlap_score(query: str, record: Dict[str, Any]) -> float:
    q_tokens = set(tokenize(query))
    if not q_tokens:
        return 0.0

    fields = [
        record.get("name", ""),
        record.get("dataset", ""),
        record.get("category", ""),
        record.get("district", ""),
        record.get("neighborhood", ""),
        record.get("address", ""),
        " ".join(record.get("keywords", []) or []),
    ]

    haystack_tokens = set()
    for field in fields:
        haystack_tokens.update(tokenize(field))

    overlap = len(q_tokens & haystack_tokens)
    return min(overlap * 0.08, 0.45)

def metadata_boost(query: str, record: Dict[str, Any]) -> float:
    q = normalize_text(query).lower()
    score = 0.0

    district = normalize_text(record.get("district")).lower()
    neighborhood = normalize_text(record.get("neighborhood")).lower()
    name = normalize_text(record.get("name")).lower()
    dataset = normalize_text(record.get("dataset")).lower()
    category = normalize_text(record.get("category")).lower()

    if district and district in q:
        score += 0.20
    if neighborhood and neighborhood in q:
        score += 0.16
    if name and name in q:
        score += 0.22
    if dataset and dataset in q:
        score += 0.10
    if category and category in q:
        score += 0.10

    return score

def is_likely_relevant(hit: SearchHit) -> bool:
    return hit.score >= 0.30

class RagCore:
    def __init__(self, client: OllamaClient):
        self.client = client
        self.records: List[Dict[str, Any]] = []

    def build_index(self, jsonl_path: str = JSONL_PATH, index_path: str = INDEX_PATH) -> None:
        raw_records: List[Dict[str, Any]] = []
        texts: List[str] = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                rec = json.loads(line)
                content = normalize_text(rec.get("content"))
                if not content:
                    continue

                raw_records.append({
                    "id": rec.get("id"),
                    "dataset": rec.get("dataset"),
                    "category": rec.get("category"),
                    "name": rec.get("name"),
                    "district": rec.get("district"),
                    "neighborhood": rec.get("neighborhood"),
                    "address": rec.get("address"),
                    "latitude": rec.get("latitude"),
                    "longitude": rec.get("longitude"),
                    "keywords": rec.get("keywords", []),
                    "attributes": rec.get("attributes", {}),
                    "content": content,
                })
                texts.append(content)

        print(f"Toplam kayıt hazırlandı: {len(raw_records)}")

        records: List[Dict[str, Any]] = []
        total = len(raw_records)

        for start in range(0, total, EMBED_BATCH_SIZE):
            end = min(start + EMBED_BATCH_SIZE, total)
            batch_records = raw_records[start:end]
            batch_texts = texts[start:end]

            embeddings = self.client.embed_many(batch_texts)

            for rec, emb in zip(batch_records, embeddings):
                rec["embedding"] = emb
                records.append(rec)

            print(f"Embedded {end}/{total}")

        with open(index_path, "wb") as f:
            pickle.dump(records, f)

        self.records = records
        print(f"Index hazır: {len(records)} kayıt")

    def load_index(self, index_path: str = INDEX_PATH) -> None:
        with open(index_path, "rb") as f:
            self.records = pickle.load(f)

    def search(self, query: str, top_k: int = TOP_K_CONTEXT) -> List[SearchHit]:
        if not self.records:
            raise RuntimeError("Index yüklenmemiş.")

        query_embedding = self.client.embed_one(query)

        scored: List[Tuple[float, Dict[str, Any]]] = []
        for rec in self.records:
            emb_score = cosine_similarity(query_embedding, rec["embedding"])
            scored.append((emb_score, rec))

        scored.sort(key=lambda x: x[0], reverse=True)
        candidates = scored[:TOP_K_CANDIDATES]

        reranked: List[SearchHit] = []
        for emb_score, rec in candidates:
            kw_score = keyword_overlap_score(query, rec)
            meta_score = metadata_boost(query, rec)
            final_score = (emb_score * 0.72) + kw_score + meta_score
            reranked.append(SearchHit(score=final_score, record=rec))

        reranked.sort(key=lambda x: x.score, reverse=True)
        reranked = [hit for hit in reranked if is_likely_relevant(hit)]
        return reranked[:top_k]

    def answer(self, query: str, hits: List[SearchHit]) -> str:
        if not hits:
            return "Bu veri setinde bulamadım."

        if hits[0].score < 0.35:
            return "Bu veri setinde bulamadım."

        context_blocks = []
        for i, hit in enumerate(hits, start=1):
            rec = hit.record
            block = {
                "sira": i,
                "score": round(hit.score, 4),
                "dataset": rec.get("dataset"),
                "category": rec.get("category"),
                "name": rec.get("name"),
                "district": rec.get("district"),
                "neighborhood": rec.get("neighborhood"),
                "address": rec.get("address"),
                "latitude": rec.get("latitude"),
                "longitude": rec.get("longitude"),
                "content": rec.get("content"),
            }
            context_blocks.append(json.dumps(block, ensure_ascii=False))

        context = "\n\n".join(context_blocks)

        prompt = f"""
Sen İstanbul şehir verileri için çalışan bir RAG asistanısın.

Kesin kurallar:
- Sadece aşağıdaki kayıtları kullan.
- Kayıtlarda açıkça olmayan bilgiyi üretme.
- Sonuçlar yetersizse veya alakasızsa sadece şu cümleyi yaz:
Bu veri setinde bulamadım.
- Tahmin yapma, genelleme yapma, dış bilgi kullanma.
- Cevabı Türkçe ver.
- Uygunsa madde madde listele.
- Varsa isim, ilçe, mahalle, adres ve koordinat bilgilerini kullan.

Kullanıcı sorusu:
{query}

Bulunan kayıtlar:
{context}

Cevap:
""".strip()

        return self.client.generate(prompt)

In [4]:
client = OllamaClient()
rag = RagCore(client)

if not os.path.exists(INDEX_PATH):
    print("Index bulunamadı, oluşturuluyor...")
    # JSONL dosyasının /content altında olduğundan emin olun!
    rag.build_index()
else:
    print("Index yükleniyor...")
    rag.load_index()

print("Sistem Kullanıma Hazır ✅")

Index bulunamadı, oluşturuluyor...
Toplam kayıt hazırlandı: 34273
Embedded 64/34273
Embedded 128/34273
Embedded 192/34273
Embedded 256/34273
Embedded 320/34273
Embedded 384/34273
Embedded 448/34273
Embedded 512/34273
Embedded 576/34273
Embedded 640/34273
Embedded 704/34273
Embedded 768/34273
Embedded 832/34273
Embedded 896/34273
Embedded 960/34273
Embedded 1024/34273
Embedded 1088/34273
Embedded 1152/34273
Embedded 1216/34273
Embedded 1280/34273
Embedded 1344/34273
Embedded 1408/34273
Embedded 1472/34273
Embedded 1536/34273
Embedded 1600/34273
Embedded 1664/34273
Embedded 1728/34273
Embedded 1792/34273
Embedded 1856/34273
Embedded 1920/34273
Embedded 1984/34273
Embedded 2048/34273
Embedded 2112/34273
Embedded 2176/34273
Embedded 2240/34273
Embedded 2304/34273
Embedded 2368/34273
Embedded 2432/34273
Embedded 2496/34273
Embedded 2560/34273
Embedded 2624/34273
Embedded 2688/34273
Embedded 2752/34273
Embedded 2816/34273
Embedded 2880/34273
Embedded 2944/34273
Embedded 3008/34273
Embedded 3

In [5]:
query = "Bağcılar'daki sağlık tesisleri"

hits = rag.search(query, top_k=8)

print("\n--- Bulunan kayıtlar ---")
for hit in hits:
    rec = hit.record
    print(
        f"[{hit.score:.4f}] "
        f"{rec.get('dataset')} | {rec.get('name')} | "
        f"{rec.get('district')} | {rec.get('address')}"
    )

print("\n--- Cevap ---")
print(rag.answer(query, hits))


--- Bulunan kayıtlar ---
[0.6062] saglik_tesisi | Bağcılar Toplum Sağlığı Merkezi Bağcılar 1 Nolu Göçmen Sağlığı Birimi | BAĞCILAR | 842. Sk. No:11 ÇINAR/BAĞCILAR
[0.5997] saglik_tesisi | Bağcılar Ağiz ve Diş Sağliği Hastanesi | BAĞCILAR | Velioğlu Cad. No:93 DEMİRKAPI/BAĞCILAR
[0.5984] saglik_tesisi | Bağcılar 35 No'lu Aile Sağlık Merkezi | BAĞCILAR | Kışla Cad. No:19 B 100. YIL/BAĞCILAR
[0.5978] saglik_tesisi | Birikim Eğitim Öğretim Ve Sağlik Hizmetleri | BAĞCILAR | <Null>
[0.5965] saglik_tesisi | Bağcılar 8 No'lu Aile Sağlığı Merkezi | BAĞCILAR | 28. Sk. No:11 BAĞLAR/BAĞCILAR
[0.5944] saglik_tesisi | Bağcılar 4 Nolu Aile Sağlığı Merkezi | BAĞCILAR | 463. Sk. No:1 A YILDIZTEPE/BAĞCILAR
[0.5942] saglik_tesisi | Bağcılar 14 No'lu Aile Sağlığı Merkezi | BAĞCILAR | Hürriyet Cad. No:49 A KİRAZLI/BAĞCILAR
[0.5930] saglik_tesisi | Bağcılar 6 No'lu Aile Sağlığı Merkezi | BAĞCILAR | 252. Sk. No:15 KAZIM KARABEKİR/BAĞCILAR

--- Cevap ---
Bağcılar'daki sağlık tesisleri şunlardır:

*   Bağcıla

In [6]:
query = "Kent lokantası"

hits = rag.search(query, top_k=8)

print("\n--- Bulunan kayıtlar ---")
for hit in hits:
    rec = hit.record
    print(
        f"[{hit.score:.4f}] "
        f"{rec.get('dataset')} | {rec.get('name')} | "
        f"{rec.get('district')} | {rec.get('address')}"
    )

print("\n--- Cevap ---")
print(rag.answer(query, hits))


--- Bulunan kayıtlar ---
[0.5494] ibb_lokasyonlar | İBB Kent Lokantası-Küçükçekmece | None | None
[0.5411] ibb_lokasyonlar | İBB Kent Lokantası-Arnavutköy | None | None
[0.5395] ibb_lokasyonlar | İBB Kent Lokantası-Çapa | None | None
[0.5359] ibb_lokasyonlar | İBB Kent Lokantası-Avcılar | None | None
[0.5354] ibb_lokasyonlar | İBB Kent Lokantası-Sultanbeyli | None | None
[0.5352] ibb_lokasyonlar | İBB Kent Lokantası-Bağcılar | None | None
[0.5347] ibb_lokasyonlar | İBB Kent Lokantası-Sultangazi | None | None
[0.5342] ibb_lokasyonlar | İBB Kent Lokantası-Çatalca | None | None

--- Cevap ---
İBB Kent Lokantası İstanbul'da şu adreslerde bulunmaktadır:

*   İBB Kent Lokantası-Küçükçekmece: Koordinatları 4540686.3322, 398576.3033999996
*   İBB Kent Lokantası-Arnavutköy: Koordinatları 4561716.5189, 394214.4528999999
*   İBB Kent Lokantası-Çapa: Koordinatları 4542728.1895, 410364.2142000003
*   İBB Kent Lokantası-Avcılar: Koordinatları 4539263.1194, 392310.24729999993
*   İBB Kent Lokantası-